# 📖 Notebook 3: Single-Table Design

In relational databases, you normalize data into many tables and JOIN them. In DynamoDB, there are **no JOINs**. Instead, the best practice is to put multiple entity types into a **single table** and design your keys around your access patterns.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why single-table design exists and when to use it
- How to model multiple entities (users, orders, items) in one table
- How to use generic key names (PK/SK) with prefixed values
- How to satisfy multiple access patterns with one table
- When single-table design is overkill

## 🛠️ Setup

Start the infrastructure first:

```bash
cd deep-dives/dynamodb
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import boto3
from boto3.dynamodb.conditions import Key, Attr
from botocore.exceptions import ClientError
import json

dynamodb = boto3.resource(
    "dynamodb",
    endpoint_url="http://localhost:8000",
    region_name="us-east-1",
    aws_access_key_id="local",
    aws_secret_access_key="local",
)

client = boto3.client(
    "dynamodb",
    endpoint_url="http://localhost:8000",
    region_name="us-east-1",
    aws_access_key_id="local",
    aws_secret_access_key="local",
)

try:
    client.list_tables()
    print("✅ Connected to DynamoDB Local")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("   Run: docker-compose up -d")

## 🤔 Why Single-Table Design?

### The SQL Way (multiple tables + JOINs)

In a relational database for an e-commerce app, you'd have:
- `users` table
- `orders` table  
- `order_items` table

To show a user's order history, you'd JOIN all three tables. Easy!

### The DynamoDB Way (no JOINs)

DynamoDB has **no JOIN operation**. If your data is in 3 tables, you need 3 separate API calls. Each call adds latency and costs money.

**Single-table design** solves this by putting all related entities in ONE table so you can fetch everything in a single Query.

### The Analogy

Think of it like organizing a file folder:
- **SQL approach**: One folder for customers, one for invoices, one for line items. To prepare a bill, you open all three folders.
- **Single-table approach**: One folder per customer. Inside, you put their profile, all their invoices, and all line items together. To prepare a bill, you open ONE folder.

## 🛒 Scenario: E-Commerce Application

We're building a simple e-commerce app. Our access patterns are:

1. **Get user profile** by user ID
2. **Get all orders** for a user (sorted by date)
3. **Get order details** including all line items
4. **Get a specific order** by order ID (using a GSI)

### Key Design Strategy

We use generic key names `PK` (partition key) and `SK` (sort key) with **prefixed values** to distinguish entity types:

| Entity | PK | SK | Example |
|--------|----|----|--------|
| User Profile | `USER#<id>` | `PROFILE` | PK=`USER#123`, SK=`PROFILE` |
| User's Order | `USER#<id>` | `ORDER#<date>#<id>` | PK=`USER#123`, SK=`ORDER#2024-01-15#ORD-001` |
| Order Item | `ORDER#<id>` | `ITEM#<num>` | PK=`ORDER#ORD-001`, SK=`ITEM#001` |

The `#` separator makes it easy to parse keys and prevents collisions.

In [ ]:
# Create the single-table with a GSI for order lookups

TABLE_NAME = "ECommerceApp"

try:
    dynamodb.Table(TABLE_NAME).delete()
    dynamodb.Table(TABLE_NAME).wait_until_not_exists()
except ClientError:
    pass

table = dynamodb.create_table(
    TableName=TABLE_NAME,
    KeySchema=[
        {"AttributeName": "PK", "KeyType": "HASH"},
        {"AttributeName": "SK", "KeyType": "RANGE"},
    ],
    AttributeDefinitions=[
        {"AttributeName": "PK", "AttributeType": "S"},
        {"AttributeName": "SK", "AttributeType": "S"},
        {"AttributeName": "GSI1PK", "AttributeType": "S"},
        {"AttributeName": "GSI1SK", "AttributeType": "S"},
    ],
    GlobalSecondaryIndexes=[
        {
            "IndexName": "GSI1",
            "KeySchema": [
                {"AttributeName": "GSI1PK", "KeyType": "HASH"},
                {"AttributeName": "GSI1SK", "KeyType": "RANGE"},
            ],
            "Projection": {"ProjectionType": "ALL"},
        },
    ],
    BillingMode="PAY_PER_REQUEST",
)

table.wait_until_exists()
print(f"✅ Created single table '{TABLE_NAME}'")
print(f"   Base table: PK (partition key) + SK (sort key)")
print(f"   GSI1: GSI1PK + GSI1SK (for alternative access patterns)")

In [ ]:
# Populate with sample data — ALL entity types in ONE table!

items = [
    # ========== User Profiles ==========
    {
        "PK": "USER#user-001", "SK": "PROFILE",
        "entity_type": "UserProfile",
        "name": "Alice Smith", "email": "alice@example.com",
        "joined": "2023-06-15",
    },
    {
        "PK": "USER#user-002", "SK": "PROFILE",
        "entity_type": "UserProfile",
        "name": "Bob Jones", "email": "bob@example.com",
        "joined": "2023-08-20",
    },

    # ========== Alice's Orders ==========
    {
        "PK": "USER#user-001", "SK": "ORDER#2024-01-10#ORD-001",
        "entity_type": "Order",
        "GSI1PK": "ORDER#ORD-001", "GSI1SK": "ORDER#ORD-001",
        "order_id": "ORD-001", "status": "delivered", "total": 89.97,
    },
    {
        "PK": "USER#user-001", "SK": "ORDER#2024-01-20#ORD-003",
        "entity_type": "Order",
        "GSI1PK": "ORDER#ORD-003", "GSI1SK": "ORDER#ORD-003",
        "order_id": "ORD-003", "status": "shipped", "total": 149.99,
    },

    # ========== Bob's Orders ==========
    {
        "PK": "USER#user-002", "SK": "ORDER#2024-01-15#ORD-002",
        "entity_type": "Order",
        "GSI1PK": "ORDER#ORD-002", "GSI1SK": "ORDER#ORD-002",
        "order_id": "ORD-002", "status": "processing", "total": 24.99,
    },

    # ========== Order Items (line items for ORD-001) ==========
    {
        "PK": "ORDER#ORD-001", "SK": "ITEM#001",
        "entity_type": "OrderItem",
        "GSI1PK": "ORDER#ORD-001", "GSI1SK": "ITEM#001",
        "product_name": "Wireless Mouse", "quantity": 1, "price": 29.99,
    },
    {
        "PK": "ORDER#ORD-001", "SK": "ITEM#002",
        "entity_type": "OrderItem",
        "GSI1PK": "ORDER#ORD-001", "GSI1SK": "ITEM#002",
        "product_name": "USB-C Cable", "quantity": 2, "price": 14.99,
    },
    {
        "PK": "ORDER#ORD-001", "SK": "ITEM#003",
        "entity_type": "OrderItem",
        "GSI1PK": "ORDER#ORD-001", "GSI1SK": "ITEM#003",
        "product_name": "Laptop Stand", "quantity": 1, "price": 29.99,
    },

    # ========== Order Items (line items for ORD-002) ==========
    {
        "PK": "ORDER#ORD-002", "SK": "ITEM#001",
        "entity_type": "OrderItem",
        "GSI1PK": "ORDER#ORD-002", "GSI1SK": "ITEM#001",
        "product_name": "Python Book", "quantity": 1, "price": 24.99,
    },

    # ========== Order Items (line items for ORD-003) ==========
    {
        "PK": "ORDER#ORD-003", "SK": "ITEM#001",
        "entity_type": "OrderItem",
        "GSI1PK": "ORDER#ORD-003", "GSI1SK": "ITEM#001",
        "product_name": "Mechanical Keyboard", "quantity": 1, "price": 149.99,
    },
]

with table.batch_writer() as batch:
    for item in items:
        batch.put_item(Item=item)

print(f"✅ Inserted {len(items)} items into single table")
print(f"   - 2 user profiles")
print(f"   - 3 orders")
print(f"   - 4 order items")
print()
print("💡 ALL of these are in the SAME table — 'ECommerceApp'")

## 🔍 Access Pattern 1: Get User Profile

To get Alice's profile, we query with `PK = USER#user-001` and `SK = PROFILE`.

This is a **GetItem** (exact key match) — the fastest possible read.

In [ ]:
# Access Pattern 1: Get user profile

response = table.get_item(
    Key={"PK": "USER#user-001", "SK": "PROFILE"}
)

profile = response["Item"]
print("👤 User Profile for Alice:")
print(f"   Name:   {profile['name']}")
print(f"   Email:  {profile['email']}")
print(f"   Joined: {profile['joined']}")

## 🔍 Access Pattern 2: Get All Orders for a User

Alice's orders all share `PK = USER#user-001` and have an `SK` that starts with `ORDER#`. We use `begins_with` on the sort key to get only order items (not the profile).

In [ ]:
# Access Pattern 2: Get all orders for a user

response = table.query(
    KeyConditionExpression=(
        Key("PK").eq("USER#user-001") &
        Key("SK").begins_with("ORDER#")
    ),
)

print("📦 Alice's Orders:")
print("=" * 60)
for item in response["Items"]:
    print(f"   {item['order_id']} | {item['status']:<12} | ${item['total']:.2f}")

print(f"\n💡 One Query fetched only orders — the PROFILE item was skipped")
print(f"   because begins_with('ORDER#') filters it out at the key level.")

## 🔍 Access Pattern 3: Get User Profile + All Orders (One Query!)

The real power of single-table design: fetch a user's profile AND all their orders in a **single Query**.

In [ ]:
# Access Pattern 3: Get EVERYTHING for a user in one query

response = table.query(
    KeyConditionExpression=Key("PK").eq("USER#user-001"),
)

print("📋 Everything for Alice (one Query):")
print("=" * 60)

for item in response["Items"]:
    entity = item["entity_type"]
    if entity == "UserProfile":
        print(f"  👤 Profile: {item['name']} ({item['email']})")
    elif entity == "Order":
        print(f"  📦 Order:   {item['order_id']} — {item['status']} — ${item['total']:.2f}")

print(f"\n📊 Total items returned: {response['Count']}")
print(f"   API calls needed: 1 (vs 2+ with separate tables)")
print()
print("💡 In SQL, this would be a JOIN across users + orders tables.")
print("   In DynamoDB single-table design, it's one Query. Faster and cheaper!")

## 🔍 Access Pattern 4: Get Order Details with Line Items (via GSI)

What if we want to look up an order by its order ID? The order lives under `PK = USER#user-001`, but the caller doesn't know which user placed it.

Our GSI (`GSI1PK = ORDER#ORD-001`) lets us find the order AND its line items in one query.

In [ ]:
# Access Pattern 4: Get order + line items by order ID (via GSI)

response = table.query(
    IndexName="GSI1",
    KeyConditionExpression=Key("GSI1PK").eq("ORDER#ORD-001"),
)

print("🛒 Order ORD-001 Details (via GSI):")
print("=" * 60)

for item in response["Items"]:
    entity = item["entity_type"]
    if entity == "Order":
        print(f"  📦 Order: {item['order_id']} | Status: {item['status']} | Total: ${item['total']:.2f}")
    elif entity == "OrderItem":
        print(f"  📎 Item:  {item['product_name']} × {item['quantity']} = ${item['price']:.2f}")

print(f"\n📊 Items returned: {response['Count']}")
print()
print("💡 The GSI groups the order header AND all its line items together.")
print("   One Query gives us the complete order — no JOINs needed!")

## 📐 Visualizing the Table Layout

Let's look at how all the data sits in one table.

In [ ]:
# Visualize the entire table

response = table.scan()
all_items = sorted(response["Items"], key=lambda x: (x["PK"], x["SK"]))

print("📋 Complete Single-Table Layout")
print("=" * 90)
print(f"{'PK':<22} {'SK':<30} {'Entity':<12} {'Details'}")
print("-" * 90)

current_pk = None
for item in all_items:
    pk = item["PK"]
    sk = item["SK"]
    entity = item.get("entity_type", "?")

    # Visual separator between partition key groups
    if pk != current_pk:
        if current_pk is not None:
            print("-" * 90)
        current_pk = pk

    details = ""
    if entity == "UserProfile":
        details = f"{item['name']} ({item['email']})"
    elif entity == "Order":
        details = f"{item['order_id']} — {item['status']} — ${item['total']:.2f}"
    elif entity == "OrderItem":
        details = f"{item['product_name']} × {item['quantity']}"

    print(f"{pk:<22} {sk:<30} {entity:<12} {details}")

print("=" * 90)
print()
print("💡 Items with the same PK are on the same partition.")
print("   USER# partitions hold profiles + orders together.")
print("   ORDER# partitions hold order items.")

## ⚖️ When to Use Single-Table Design

### ✅ Use it when:
- You have well-defined access patterns that won't change often
- You need to fetch related entities in a single round-trip
- Performance and cost efficiency matter (fewer API calls = less latency and cost)

### ❌ Avoid it when:
- Access patterns are unknown or change frequently (early-stage startup)
- Your team is unfamiliar with DynamoDB (the learning curve is steep)
- You need complex ad-hoc queries (use a relational database instead)
- Data relationships are complex and fluid

### The Trade-off
Single-table design trades **data modeling complexity** for **runtime efficiency**. You invest more time upfront designing your keys, but your queries are faster and cheaper at runtime.

## 🎯 Key Takeaways

1. **Single-table design** puts multiple entity types in one table to avoid multiple API calls
2. Use **generic key names** (PK/SK) with **prefixed values** (USER#, ORDER#, ITEM#)
3. `begins_with()` on the sort key lets you filter entity types within a partition
4. **GSIs** provide alternative access patterns (e.g., look up order by ID)
5. **Design your keys around access patterns**, not around entities
6. It's a trade-off: more complex data modeling, but faster and cheaper queries

### Design Process
1. List all your **access patterns** first
2. Design your **PK and SK** to satisfy the most common patterns
3. Add **GSIs** for patterns that need a different partition key
4. Choose **projection types** to minimize storage costs

### Next Up
In the next notebook, we'll learn about **DynamoDB Streams and CDC** — how to react to data changes in real time.